# Dataset Preparation — CONFLICTS 621

Prepare a five-class classification dataset containing 458 expert-labelled CONFLICTS examples, 110 synthetic Misinformation examples, and 53 synthetic Outdated examples. The dataset contains 621 examples and has a fixed record order for reproducible splitting.


## 1. Load and validate source data

Read `conflicts_458.jsonl`, `misinformation_110.jsonl`, and `outdated_53.jsonl` from the input directory. Preserve the question, document titles, document short texts, and conflict labels; validate the combined file and save it as `conflicts_621.jsonl`.


In [1]:
# Input and output directories for local and Kaggle execution
from pathlib import Path
import os

KAGGLE_INPUT = Path("/kaggle/input/conflicts")
IS_KAGGLE = Path("/kaggle/input").is_dir()
if IS_KAGGLE:
    if not KAGGLE_INPUT.is_dir():
        raise FileNotFoundError("Attach your private 'conflicts' dataset using Kaggle Add Input first.")
    DATA_DIR = KAGGLE_INPUT
    OUTPUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = Path.cwd()
    OUTPUT_DIR = Path.cwd()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(OUTPUT_DIR)  # Save generated files in the writable output directory
print("Input directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

import json
import pandas as pd

Input directory: /Users/GXM/03-trimester 2/AIML339/Assignment/6
Output directory: /Users/GXM/03-trimester 2/AIML339/Assignment/6


In [2]:
# Load and merge the original and synthetic datasets.
# Keep the required classification fields and validate all records.

ORIGINAL_PATH = str(DATA_DIR / "conflicts_458.jsonl")
MISINFO_PATH = str(DATA_DIR / "misinformation_110.jsonl")
OUTDATED_PATH = str(DATA_DIR / "outdated_53.jsonl")
# Combined dataset 
OUTPUT_PATH = str(OUTPUT_DIR / "conflicts_621.jsonl")
def load_jsonl(file_path):
    """Load a JSONL file and report malformed JSON with its line number."""
    records = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSON in {file_path}, line {line_number}: {e}"
                )

    return records


def project_original_example(item):
    """
    Keep the required classification fields from an original CONFLICTS example.
    """
    return {
        "question": item["question"],
        "search_results": [
            {
                "title": doc.get("title", "") or "",
                "short_text": doc.get("short_text", "") or "",
            }
            for doc in item["search_results"]
        ],
        "conflict_type": item["conflict_type"],
    }


def validate_final_example(item, dataset_name, example_index):
    """
    Validate a synthetic example without changing its content.
    """
    expected_top_fields = {
        "question",
        "search_results",
        "conflict_type",
    }

    expected_doc_fields = {
        "title",
        "short_text",
    }

    assert set(item.keys()) == expected_top_fields, (
        f"{dataset_name} example {example_index}: "
        f"unexpected top-level fields {set(item.keys())}"
    )

    assert isinstance(item["question"], str)
    assert item["question"].strip()

    assert isinstance(item["search_results"], list)
    assert len(item["search_results"]) > 0

    assert isinstance(item["conflict_type"], str)
    assert item["conflict_type"].strip()

    for doc_index, doc in enumerate(item["search_results"]):
        assert set(doc.keys()) == expected_doc_fields, (
            f"{dataset_name} example {example_index}, document {doc_index}: "
            f"unexpected fields {set(doc.keys())}"
        )

        assert isinstance(doc["title"], str)
        assert isinstance(doc["short_text"], str)
# 1. Load the three separate datasets
original_raw = load_jsonl(ORIGINAL_PATH)
misinfo_final = load_jsonl(MISINFO_PATH)
outdated_final = load_jsonl(OUTDATED_PATH)

print("Loaded datasets:")
print("  Original CONFLICTS:", len(original_raw))
print("  Synthetic Misinformation:", len(misinfo_final))
print("  Synthetic Outdated:", len(outdated_final))
# 2. Check source dataset sizes
assert len(original_raw) == 458, (
    f"Expected 458 original examples, found {len(original_raw)}"
)

assert len(misinfo_final) == 110, (
    f"Expected 110 Misinformation examples, found {len(misinfo_final)}"
)

assert len(outdated_final) == 53, (
    f"Expected 53 Outdated examples, found {len(outdated_final)}"
)
# 3. Prepare original CONFLICTS records
original_clean = [
    project_original_example(item)
    for item in original_raw
]

# Validate the projected original examples.
for i, item in enumerate(original_clean, start=1):
    validate_final_example(
        item,
        dataset_name="Original CONFLICTS",
        example_index=i,
    )
# 4. Validate synthetic records
for i, item in enumerate(misinfo_final, start=1):
    validate_final_example(
        item,
        dataset_name="Misinformation",
        example_index=i,
    )

for i, item in enumerate(outdated_final, start=1):
    validate_final_example(
        item,
        dataset_name="Outdated",
        example_index=i,
    )


# Check synthetic labels.
assert all(
    item["conflict_type"] == "Conflict due to misinformation"
    for item in misinfo_final
)

assert all(
    item["conflict_type"] == "Conflict due to outdated information"
    for item in outdated_final
)
# 5. Merge the 621 examples
data = (
    original_clean
    + misinfo_final
    + outdated_final
)

assert len(data) == 621
# 6. Validate class distribution
label_counts = pd.Series(
    [item["conflict_type"] for item in data]
).value_counts()

expected_counts = {
    "No conflict": 161,
    "Complementary information": 115,
    "Conflicting opinions and research outcomes": 115,
    "Conflict due to outdated information": 115,
    "Conflict due to misinformation": 115,
}

print("\nClass distribution:")
print(label_counts)

for label, expected_count in expected_counts.items():
    actual_count = int(label_counts.get(label, 0))

    assert actual_count == expected_count, (
        f"{label}: expected {expected_count}, found {actual_count}"
    )
# 7. Save the merged classification dataset
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for item in data:
        f.write(
            json.dumps(item, ensure_ascii=False)
            + "\n"
        )
# 8. Reload and verify the saved dataset
reloaded_data = load_jsonl(OUTPUT_PATH)

assert len(reloaded_data) == 621
assert reloaded_data == data

print("Total examples:", len(data))
print("Saved to:", OUTPUT_PATH)



Loaded datasets:
  Original CONFLICTS: 458
  Synthetic Misinformation: 110
  Synthetic Outdated: 53

Class distribution:
No conflict                                   161
Complementary information                     115
Conflicting opinions and research outcomes    115
Conflict due to misinformation                115
Conflict due to outdated information          115
Name: count, dtype: int64
Total examples: 621
Saved to: /Users/GXM/03-trimester 2/AIML339/Assignment/6/conflicts_621.jsonl


## 2. Inspect class counts and sample records


In [3]:
print(data[0].keys())
print()

print("Question:")
print(data[0]["question"])

print("\nConflict type:")
print(data[0]["conflict_type"])

print("\nNumber of search results:")
print(len(data[0]["search_results"]))

dict_keys(['question', 'search_results', 'conflict_type'])

Question:
who is commander chief of the military

Conflict type:
Complementary information

Number of search results:
10


In [4]:
example_doc = data[0]["search_results"][0]

print("Document keys:")
print(example_doc.keys())

print("\nTitle:")
print(example_doc["title"])

print("\nShort text:")
print(example_doc["short_text"])

Document keys:
dict_keys(['title', 'short_text'])

Title:
Commander-in-chief - Wikipedia

Short text:
Accordingly, the Monarch is commander-in-chief of the Norwegian Armed Forces, and serves as chief diplomatic official abroad and as a symbol of unity.
 The President of the Republic, for a period of war, shall appoint the Commander-in-Chief of the Armed Forces on request of the Prime Minister. He may dismiss the Commander-in-Chief of the Armed Forces in accordance with the same procedure. The authority of the Commander-in-Chief of the Armed Forces, as well as the principle of his subordination to the constitutional organs of the Republic of Poland, shall be specified by statute.
 In the Portuguese military parlance, the term "Commander-in-Chief" (in Portuguese: comandante-em-chefe or simply comandante-chefe) refers to the unified military commander of all the land, naval and air forces in a theater of operations.
 Rwanda
Saudi Arabia
Article 60 of the Basic Law of Saudi Arabia states: 

## 3. Classification dataset

`conflicts_621.jsonl` contains the records used by all classification experiments. Its record order determines the indices stored in the split manifest. The model notebooks load the dataset and manifest as a pair.


## 4. Group-aware Train / Validation / Test split

Create `conflicts_original_eval_split.json` using a deterministic, group-aware split of the 458 original CONFLICTS examples. Grouping keeps identical questions and shared substantial document texts in the same split. All 163 synthetic examples belong to Training; Validation and Test contain only original CONFLICTS examples.

The split manifest stores record indices and a dataset SHA-256 checksum. Exact-match checks cover questions and retrieved-document text; the code does not verify semantic near-duplicates or factual label correctness.


In [5]:
# Create a shared group-aware split with original-only validation and test data.
# Dataset order: 458 CONFLICTS + 110 synthetic Misinformation + 53 synthetic Outdated.
# Manifest indices refer to record positions in conflicts_621.jsonl.
import hashlib
import re

SPLIT_SEED = 42
SPLIT_MANIFEST_PATH = str(OUTPUT_DIR / "conflicts_original_eval_split.json")
def normalized_question(record):
    return " ".join(record["question"].casefold().split())

def fingerprint_records(records):
    canonical = "\n".join(json.dumps(x, ensure_ascii=False, sort_keys=True) for x in records)
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()

assert len(data) == 621
original_records, synthetic_records = data[:458], data[458:]
assert pd.Series([x["conflict_type"] for x in original_records]).value_counts().to_dict() == {
    "No conflict": 161,
    "Complementary information": 115,
    "Conflicting opinions and research outcomes": 115,
    "Conflict due to outdated information": 62,
    "Conflict due to misinformation": 5,
}
assert pd.Series([x["conflict_type"] for x in synthetic_records]).value_counts().to_dict() == {
    "Conflict due to outdated information": 53,
    "Conflict due to misinformation": 110,
}

# Group original examples that share an exact substantial retrieved-document body.
# A connected component cannot be divided across Train / Val / Test.
parent = list(range(458))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

seen_bodies = {}
seen_questions = {}
for idx, item in enumerate(original_records):
    q = normalized_question(item)
    if q in seen_questions:
        union(idx, seen_questions[q])
    seen_questions[q] = idx
    for doc in item["search_results"]:
        body = " ".join((doc.get("short_text") or "").casefold().split())
        if len(body) >= 120:
            h = hashlib.sha256(body.encode("utf-8")).hexdigest()
            if h in seen_bodies:
                union(idx, seen_bodies[h])
            seen_bodies[h] = idx

from collections import defaultdict, Counter
components = defaultdict(list)
for idx in range(458):
    components[find(idx)].append(idx)

classes = sorted({x["conflict_type"] for x in original_records})
totals = Counter(x["conflict_type"] for x in original_records)
targets = {}
for label, n in totals.items():
    if n == 5:
        targets[label] = (3, 1, 1)
    else:
        t, v = int(n * 0.70), int(n * 0.85) - int(n * 0.70)
        targets[label] = (t, v, n-t-v)

# Group-aware greedy stratification. Fixed seed, deterministic order.
import random
rng = random.Random(SPLIT_SEED)
groups = list(components.values())
rng.shuffle(groups)
groups.sort(key=len, reverse=True)
assigned = [[], [], []]
counts = [Counter(), Counter(), Counter()]
for group in groups:
    gcounts = Counter(original_records[i]["conflict_type"] for i in group)
    scores = []
    for split in range(3):
        cost = 0.0
        for label in classes:
            target = max(1, targets[label][split])
            after = counts[split][label] + gcounts[label]
            overflow = max(0, after - target)
            cost += (overflow / target)**2 * 10
            cost -= min(gcounts[label], max(0, target-counts[split][label])) / target
        scores.append(cost)
    k = min(range(3), key=lambda j: (scores[j], len(assigned[j])))
    assigned[k].extend(group)
    counts[k].update(gcounts)
orig_train, orig_val, orig_test = assigned
assert all(len(x)>0 for x in assigned)
print("Original split class counts (group-aware, approximate 70/15/15):")
for label in classes:
    print(label, [counts[k][label] for k in range(3)])

# Check exact cross-split overlaps in questions and document texts.
train_ids = orig_train + list(range(458, 621))
val_ids, test_ids = orig_val, orig_test
assert len(set(train_ids + val_ids + test_ids)) == 621
assert set(train_ids).isdisjoint(val_ids) and set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)

def question_set(ids):
    return {normalized_question(data[i]) for i in ids}
assert not (question_set(train_ids) & question_set(val_ids))
assert not (question_set(train_ids) & question_set(test_ids))
assert not (question_set(val_ids) & question_set(test_ids))

def document_hashes(ids):
    hashes = set()
    for i in ids:
        for d in data[i]["search_results"]:
            txt = " ".join((d.get("short_text") or "").casefold().split())
            if len(txt) >= 120:
                hashes.add(hashlib.sha256(txt.encode("utf-8")).hexdigest())
    return hashes
trh, vah, teh = document_hashes(train_ids), document_hashes(val_ids), document_hashes(test_ids)
assert not (trh & vah or trh & teh or vah & teh), "Exact cross-split document duplicate detected"

split_manifest = {
    "split_seed": SPLIT_SEED,
    "dataset_sha256": fingerprint_records(data),
    "original_count": 458,
    "synthetic_count": 163,
    "train_indices": sorted(train_ids),
    "validation_indices": sorted(val_ids),
    "test_indices": sorted(test_ids),
    "original_train_indices": sorted(orig_train),
    "synthetic_train_indices": list(range(458, 621)),
    "note": "Synthetic examples are training-only. No human taxonomy review or semantic near-duplicate guarantee is implied."
}
with open(SPLIT_MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump(split_manifest, f, ensure_ascii=False, indent=2)
print("Saved", SPLIT_MANIFEST_PATH)
for split in ("train", "validation", "test"):
    ids = split_manifest[f"{split}_indices"]
    print(split, len(ids), "original:", sum(i < 458 for i in ids), "synthetic:", sum(i >= 458 for i in ids))


Original split class counts (group-aware, approximate 70/15/15):
Complementary information [80, 17, 18]
Conflict due to misinformation [3, 1, 1]
Conflict due to outdated information [43, 9, 10]
Conflicting opinions and research outcomes [80, 17, 18]
No conflict [112, 24, 25]
Saved /Users/GXM/03-trimester 2/AIML339/Assignment/6/conflicts_original_eval_split.json
train 481 original: 318 synthetic: 163
validation 68 original: 68 synthetic: 0
test 72 original: 72 synthetic: 0
